# Lab: Distribution Plots

This lab reinforces the techniques from Lesson 4 — histograms, KDE curves, violin plots,
and ECDF plots — using the same retail sales dataset introduced in Lesson 1.

Work through each section in order. The questions at the end of each step are prompts
for reflection; write your answers in the empty markdown cells provided.

**Outputs are cleared.** Run each cell to generate the plots.

## Setup: install dependencies and build the dataset

In [ ]:
!pip install matplotlib seaborn pandas numpy --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.0)

np.random.seed(42)
n = 500

df = pd.DataFrame({
    "date": pd.date_range("2023-01-01", periods=n, freq="D"),
    "category": np.random.choice(["Electronics", "Clothing", "Books", "Home"], n),
    "region": np.random.choice(["North", "South", "East", "West"], n),
    "sales": np.random.lognormal(mean=6.5, sigma=0.8, size=n).round(2),
    "units": np.random.randint(1, 20, n),
    "customer_age": np.random.normal(38, 10, n).clip(18, 70).astype(int),
    "profit_margin": np.random.beta(3, 7, n).round(3),
})

print(df.shape)
df.head()

## Step 1: Histogram with KDE overlay

Plot the distribution of `sales` with both bars and a KDE line.
Try different `bins` values (20, 40, 80) and observe how the bar shape changes
while the KDE stays stable.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, bins in zip(axes, [20, 40, 80]):
    sns.histplot(df["sales"], bins=bins, kde=True, ax=ax)
    ax.set_title(f"{bins} bins")
    ax.set_xlabel("Sales (£)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

axes[0].set_ylabel("Count")
fig.suptitle("Sales distribution — bin count comparison", fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

> **Question:** At which bin count does the distribution look most informative?
> Does the number of modes (peaks) change as you vary the bin count?
> What does the KDE tell you that the bars can't?

*(Write your answer here.)*

## Step 2: KDE with rugplot

Plot KDE + rug for `profit_margin`. The profit margin variable has a different
shape from sales — it's bounded between 0 and 1 and follows a beta distribution.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.kdeplot(df["profit_margin"], fill=True, ax=ax, color="#dd8452", alpha=0.4)
sns.rugplot(df["profit_margin"], ax=ax, height=0.04, color="#dd8452", alpha=0.3)
ax.set_title("Profit margin density — most transactions cluster below 0.5")
ax.set_xlabel("Profit margin")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()

> **Question:** Where does the profit margin peak? Does the KDE extend below 0?
> If so, what does that mean, and is it a problem?

*(Write your answer here.)*

## Step 3: Overlaid KDEs by category

Overlay the sales KDE for each product category on a single axis.
Use `fill=True` with low alpha so overlapping areas stay readable.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for cat in sorted(df["category"].unique()):
    subset = df[df["category"] == cat]
    sns.kdeplot(subset["sales"], fill=True, label=cat, alpha=0.3, ax=ax)

ax.set_title("Sales distributions by category")
ax.set_xlabel("Sales (£)")
ax.legend(title="Category", frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()

> **Question:** Are the distributions for each category similar or clearly different?
> Which category tends to have the highest-value sales?
> Would a histogram work as well as KDE here for comparing four groups? Why or why not?

*(Write your answer here.)*

## Step 4: Violin plots

Build a violin plot of `sales` by `region`. Order the violins by median sales
(highest to lowest) so the ranking is immediately readable.

In [ ]:
order = (
    df.groupby("region")["sales"]
    .median()
    .sort_values(ascending=False)
    .index
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.violinplot(data=df, x="region", y="sales", order=order, ax=ax)
ax.set_title("Sales distribution by region (ordered by median)")
ax.set_xlabel("")
ax.set_ylabel("Sales (£)")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()

> **Question:** Do any regions have noticeably different distribution shapes?
> Can you see any bimodality (two humps) in any violin?
> What does the width of the violin at a given y-value mean?

*(Write your answer here.)*

## Step 5: ECDF comparison

Plot ECDF curves for `sales` split by `region`. Add a horizontal dashed line
at y=0.5 to mark the median for each group.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.ecdfplot(data=df, x="sales", hue="region", ax=ax)
ax.axhline(0.5, color="0.6", linestyle="--", linewidth=0.8, label="Median (p50)")
ax.set_title("Sales ECDF by region — read medians at the 0.5 line")
ax.set_xlabel("Sales (£)")
ax.set_ylabel("Proportion ≤ x")
ax.legend(title="Region", frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()

> **Question:** Using the dashed line, estimate the median sales for each region.
> Which region has the steepest ECDF curve, and what does that imply about
> the spread of its sales values?

*(Write your answer here.)*

## Challenge: choose the right plot

You need to answer the following question:

> *"Do customers in different age groups have meaningfully different profit margins?"*

Create an age group column by binning `customer_age` into three groups:
`18–34`, `35–54`, `55+`. Then choose **one** of the distribution plot types
from this lesson to answer the question. Add a title that states your finding.

In [ ]:
df["age_group"] = pd.cut(
    df["customer_age"],
    bins=[17, 34, 54, 70],
    labels=["18–34", "35–54", "55+"]
)

# Your plot here — pick histplot, kdeplot, violinplot, or ecdfplot


> **Question:** Which plot type did you choose, and why was it the best fit for this question?
> What does your chart tell you about the relationship between age group and profit margin?

*(Write your answer here.)*